# 13 — Extension aux paires d'ETF (réplication du pipeline actions)

**Pourquoi les ETF (Chan, chap. 4, p. 89-92).** Les paires d'actions se défont hors échantillon parce que les fondamentaux d'une **entreprise** changent vite (management, dette, produits). L'économie d'un **panier** (ETF) change lentement : les paires d'ETF restent donc cointégrées plus longtemps. C'est l'extension naturelle du projet.

**Règle de conception (Chan, p. 92) — physique vs futures.** Un ETF sur **futures** de matières premières (ex. USO sur le pétrole) ne détient pas le sous-jacent : à cause du *roll return*, il ne cointègre pas forcément avec les producteurs. On privilégie les ETF qui détiennent le **physique** (GLD = or physique) plutôt que des futures (USO).

**Candidats économiquement motivés (Chan) :** EWA/EWC (ETF pays, économies de matières premières), GLD/GDX (or physique vs mineurs d'or), RTH/XLP (retail vs consommation de base).

Ce notebook **réutilise à l'identique** `src/screening.py` (cointégration, Johansen, Hurst, VR, record-every-backtest, DSR, calibration OU) et `src/backtest_stats.py` (TuW, HHI, hit ratio, P[échec]). Seul l'univers change.

> ⚠️ À exécuter en local (téléchargement `yfinance`).

In [1]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, yfinance as yf
import screening as S, backtest_stats as BS

# Univers ETF : candidats de Chan + quelques ETF sectoriels pour un petit screen intra-groupe.
# 'group' = thèse économique (l'équivalent du "secteur" pour la génération de candidates).
etf_groups = pd.Series({
    "EWA": "Pays_MatPrem", "EWC": "Pays_MatPrem",          # Australie / Canada
    "GLD": "Or_physique",  "IAU": "Or_physique",           # or physique
    "GDX": "Or_mineurs",   "GDXJ": "Or_mineurs",           # mineurs d'or
    "RTH": "Conso",        "XLP": "Conso",  "XLY": "Conso", # retail / conso de base / conso discr.
    "XLE": "Energie",      "XOP": "Energie",               # énergie
    "USO": "Petrole_futures",                              # contre-exemple roll return (futures)
})
tickers = list(etf_groups.index)
etf = yf.download(tickers, start="2011-01-01", auto_adjust=True)["Close"].dropna(how="all")
etf = etf.dropna()   # période commune à tous les ETF
etf.to_csv("../data/etf_adj_close.csv"); etf_groups.to_frame("secteur").to_csv("../data/etf_sectors.csv", index_label="ticker")
print("Univers ETF :", etf.shape[1], "ETF |", etf.index.min().date(), "->", etf.index.max().date())

[*********************100%***********************]  12 of 12 completed


Univers ETF : 12 ETF | 2011-01-03 -> 2026-08-31


## 1. Paires-thèses de Chan : cointégration et mean-reversion

On teste directement les trois paires économiquement motivées, avec le **même trio** que sur les actions : Johansen (cointégration symétrique), Hurst et ratio de variance (mean-reversion du spread).

In [2]:
for A, B in [("EWA","EWC"), ("GLD","GDX"), ("RTH","XLP")]:
    if A not in etf or B not in etf: 
        print(f"{A}/{B} : données manquantes"); continue
    beta, trace, cv = S.johansen_beta(etf, A, B)
    spread = etf[A] - beta * etf[B]
    H = S.hurst_exponent(spread.values); VR = S.variance_ratio(spread.values, 20)
    p_eg = S.eg_pvalue(etf, A, B)
    print(f"{A}/{B:4s} | EG p={p_eg:.3f} | Johansen trace {trace:.1f} vs {cv:.1f} "
          f"({'cointégré' if trace>cv else 'NON'}) | β={beta:.3f} | Hurst {H:.2f} | VR {VR:.2f}")

EWA/EWC  | EG p=0.133 | Johansen trace 19.0 vs 15.5 (cointégré) | β=0.338 | Hurst 0.41 | VR 0.74
GLD/GDX  | EG p=0.102 | Johansen trace 13.0 vs 15.5 (NON) | β=4.207 | Hurst 0.36 | VR 0.73
RTH/XLP  | EG p=0.091 | Johansen trace 12.3 vs 15.5 (NON) | β=3.757 | Hurst 0.43 | VR 0.90


**Règle physique vs futures (contre-exemple).** On compare une paire or-physique/mineurs (GLD/GDX) à une paire qui inclut un ETF **futures** (USO). Le roll return doit dégrader la cointégration de la seconde.

In [3]:
for A, B in [("GLD","GDX"), ("USO","XLE"), ("USO","GDX")]:
    if A not in etf or B not in etf: continue
    p_eg = S.eg_pvalue(etf, A, B)
    _, trace, cv = S.johansen_beta(etf, A, B)
    print(f"{A}/{B:4s} | EG p={p_eg:.3f} | Johansen {'cointégré' if trace>cv else 'NON cointégré (roll return ?)'}")

GLD/GDX  | EG p=0.102 | Johansen NON cointégré (roll return ?)
USO/XLE  | EG p=0.646 | Johansen NON cointégré (roll return ?)
USO/GDX  | EG p=0.688 | Johansen NON cointégré (roll return ?)


## 2. Screening complet + DSR rigoureux (record every backtest)

On applique le **pipeline identique** aux actions : screening de cointégration intra-groupe, puis DSR déflaté sur tous les essais. Avantage attendu des ETF (Chan) : **moins d'essais** (univers plus petit, paires motivées) → moins de biais de sélection, et cointégration plus durable.

In [4]:
ranked, n_cand = S.screen_cointegration(etf, etf_groups, corr_min=0.30)
ranked.to_csv("../data/etf_pairs_ranked.csv", index=False)
print(f"Candidates ETF testées : {n_cand} | cointégrées après BH : {len(ranked)}")
display(ranked[["paire","secteur","p_bh","half_life","beta_cv","SCORE"]].round(3))

trials = S.record_all_trials(etf, etf_groups, corr_min=0.30)
years = len(etf) / 252
if len(ranked):
    best = ranked.iloc[0]["paire"]; a, b = best.split("/")
    _, bets = S.vec_pair_backtest(etf, a, b)
    d = S.deflated_sharpe(trials["theta"], bets, years)
    print(f"\nMeilleure paire (SCORE) : {best} | DSR = {d['dsr']:.3f} (N={d['N']}, SR*={d['sr_star']:.2f})")

Candidates ETF testées : 7 | cointégrées après BH : 3


,paire,secteur,p_bh,half_life,beta_cv,SCORE
0,GDX/GDXJ,Or_mineurs,0.014,434.901,0.281,0.567
1,RTH/XLY,Conso,0.014,85.711,0.342,0.545
2,XLP/XLY,Conso,0.014,92.304,0.787,0.508



Meilleure paire (SCORE) : GDX/GDXJ | DSR = 0.869 (N=7, SR*=0.30)


## 3. Calibration OU (seuils + stop-loss) et backtest avec coûts

Même calibration triple-barrière sur chemins synthétiques (chap. 13) et même backtest vectorisé avec coûts que sur les actions.

In [5]:
if len(ranked):
    a, b = ranked.iloc[0]["paire"].split("/")
    grid, ou = S.calibrate_ou_thresholds(etf, a, b)
    print(f"{a}/{b} — demi-vie {ou['half_life']:.0f} j")
    daily, bets = S.vec_pair_backtest(etf, a, b)
    sr = daily.mean()/daily.std()*np.sqrt(252)
    print(f"Sharpe (entrée 2.5 / sortie 0.5, coûts) : {sr:.3f}")

GDX/GDXJ — demi-vie 435 j
Sharpe (entrée 2.5 / sortie 0.5, coûts) : 0.392


## 4. Statistiques de risque (chap. 14 / 15)

Mêmes mesures que sur les actions : Time under Water, hit ratio, probabilité d'échec.

In [6]:
if len(ranked):
    a, b = ranked.iloc[0]["paire"].split("/")
    daily, bets = S.vec_pair_backtest(etf, a, b)
    dd, tuw = BS.drawdown_tuw(daily); h = BS.hit_stats(bets)
    print(f"TuW 95e : {tuw.quantile(.95):.2f} an | DD 95e : {dd.quantile(.95):.1%}")
    print(f"hit ratio : {h['hit_ratio']:.1%} | {h['n_paris']} paris")
    freq = len(bets)/years
    pf = BS.prob_failure(bets, freq, 1.0)
    print(f"P[échec Sharpe 1.0] : {pf['prob_echec']:.1%}")

TuW 95e : 0.89 an | DD 95e : 7.0%
hit ratio : 69.2% | 39 paris
P[échec Sharpe 1.0] : 65.9%


## 5. Récit de rupture de cointégration : GLD/GDX en 2008 (Chan, p. 91-92)

La cointégration GLD/GDX **se brise en juillet 2008** : quand le pétrole grimpe, miner de l'or coûte plus cher, donc GDX décroche de GLD. La démarche de Chan — **former une hypothèse, la tester, ajouter USO pour former un triplet qui recointègre** — est la méthode scientifique appliquée au trading. À rédiger dans la section « rupture de cointégration » du rapport, en testant empiriquement la cointégration GLD/GDX **avant** vs **après** mi-2008 (sous-échantillons), puis le triplet GLD/GDX/USO.

**Conclusion attendue de l'extension ETF :** paires plus stables dans le temps, moins d'essais (donc DSR plus favorable), mais rendement de réversion plus faible — cohérent avec l'idée que la robustesse se paie en performance.